# Phase1: データ拡張の可視化

**この回で学ぶこと**
- 反転・回転・切り抜き・色調変更などの拡張が、画像をどう変えるか
- 拡張は**ランダム**で、同じ操作でも毎回違う結果になること
- 拡張の**強さ**（パラメータ）による見た目の違い

学習の前に、拡張した画像を**自分の目で確認する**のがコツです。ラベルが変わってしまう（ありえない画像になる）拡張は使えません。

> ⏱ 所要時間: 1分

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"torch が見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (image_augment_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

In [ ]:
import torch, random
from torchvision.transforms import v2

random.seed(0); torch.manual_seed(0)
files = sorted((DATA / "train" / "ants").glob("*.jpg"))
img = v2.Resize((224, 224))(Image.open(files[0]).convert("RGB"))
img

## 1. 拡張ギャラリー

`src/visualize_augment.py` に定義済みの10種類を、1枚の画像に1つずつ適用します。

In [ ]:
import visualize_augment as V

items = [("元画像", img)] + [(name, aug(img)) for name, aug in V.AUGMENTS.items()]
show_images([im for _, im in items], [n for n, _ in items], cols=4);

**観察ポイント**
- 回転・アフィン・透視変換は、**角が黒く埋まる**。黒枠を手がかりに学習してしまわないか？
- 画像には撮影ID（UGA…）の文字がある。反転すると鏡文字に。ラベルと無関係な文字を手がかりにしていないか？
- 色調変更は背景の青を大きく変える。アリ/ハチの判別に色は必要か？

## 2. 拡張は「ランダム」

同じ `RandomResizedCrop` を8回適用すると、毎回違う画像になります。学習ではエポックごとに違う見え方の画像を使うため、1枚の画像が**実質的に何枚分もの学習データ**になります。

In [ ]:
aug = v2.RandomResizedCrop(224, scale=(0.4, 1.0))
show_images([aug(img) for _ in range(8)], [f"{i+1}回目" for i in range(8)], cols=4);

## 3. 拡張の「強さ」を変えてみる

強すぎると被写体が欠けたり不自然になり、学習が進みません。**弱めから始める**のが基本です。

In [ ]:
def strength_row(title, make, levels):
    ims = [make(l)(img) for l in levels]
    show_images(ims, [f"{title}={l}" for l in levels], cols=len(levels), size=2.6)

strength_row("回転degrees", lambda d: v2.RandomRotation(degrees=(d, d)), [10, 30, 60, 90])
strength_row("切り抜きscale下限", lambda s: v2.RandomResizedCrop(224, scale=(s, s)), [0.8, 0.5, 0.2, 0.08])
strength_row("色調の強さ", lambda k: v2.ColorJitter(k, k, k, min(k, 0.5)), [0.1, 0.3, 0.6, 0.9])
strength_row("ぼかしsigma", lambda s: v2.GaussianBlur(9, sigma=(s, s)), [0.5, 1.5, 3.0, 5.0]);

## 4. 拡張を組み合わせる（`Compose`）

実際の学習では複数の拡張を順番に適用します。この後の実験で使う `basic` は **切り抜き → 反転 → 色調** の組み合わせです。

In [ ]:
basic = v2.Compose(T.AUG_SETS["basic"])
show_images([basic(Image.open(files[0]).convert("RGB")) for _ in range(8)], [f"basic {i+1}" for i in range(8)], cols=4);

## 🧪 試してみよう
1. `RandomRotation` を `degrees=180` にすると何が起きる？ アリ/ハチの判別に向いているか？
2. `files[0]` を `bees` の画像に替えて、同じ拡張を試そう。色調変更の影響は違って見える？
3. **ラベルを壊す拡張を考えてみよう。** もし分類対象が「手書き数字の6と9」だったら、どの拡張が危険か？（→ 回転・上下反転）
4. `T.AUG_SETS` に自分の組み合わせを追加するなら、どの順序が自然か？（幾何変換 → 色変換 → テンソル化 → 正規化）

## まとめ
- 拡張は学習時に**ランダム**に適用され、1枚の画像から多様な見え方を作る。
- 強さは弱めから。**拡張後の画像が人間にも同じクラスに見える**ことが条件。
- 学習前に必ず可視化して確認する。

➡ 次は `02_baseline.ipynb`（拡張なしで学習し、過学習を確認する）